#Imports

In [1]:
import tarfile
import os
import matplotlib.pyplot as plt
from wordcloud import WordCloud

import torch
import torch.nn as nn
from torch.optim import AdamW, Adam
from datasets import load_dataset
from transformers import BartForConditionalGeneration, BartTokenizer

from tqdm import tqdm


# Universal Variable

For reproducability

## Setting up Google Drive

In [2]:
from google.colab import drive
drive.mount('/content/drive')

Mounted at /content/drive


## Set device to cuda if available

In [3]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

## Setting Random Seed

In [4]:
RANDOM_SEED = 42

torch.manual_seed(RANDOM_SEED)
torch.cuda.manual_seed(RANDOM_SEED)

## Links/ Paths

In [5]:
zippedDataSetPath = '/content/drive/MyDrive/Natural_Language_Processing/Dataset/english_XLSum_v2.0.tar.bz2'
extractedDataSetPath = '/content/xlsum_en_dataset'

modelSavePath = '/content/drive/MyDrive/Natural_Language_Processing/Model_Saves'

## Hyperparameters

In [6]:
MAX_SUMMARY_LENGTH = 64
MAX_INPUT_LENGTH = 512

BEAM_SEARCH_WIDTH = 4

BATCH_SIZE = 256
EPOCHS = 1

LEARNING_RATE = 1e-5

# Extracting and Loading Dataset

In [7]:
# Create Directory if it does not exist
os.makedirs(extractedDataSetPath, exist_ok=True)

# Extract the tar file
with tarfile.open(zippedDataSetPath, 'r:bz2') as tar:
    tar.extractall(path=extractedDataSetPath)

print("Files have been successfully extracted.")

/tmp/ipykernel_1840/1810096438.py:6: DeprecationWarning: Python 3.14 will, by default, filter extracted tar archives and reject files or modify their metadata. Use the filter argument to control this behavior.
  tar.extractall(path=extractedDataSetPath)


Files have been successfully extracted.


In [8]:
xlsumEnDataset = load_dataset('json', data_files={
    'test': os.path.join(extractedDataSetPath, 'english_test.jsonl'),
    'val': os.path.join(extractedDataSetPath, 'english_val.jsonl')
})

Generating test split: 0 examples [00:00, ? examples/s]

Generating val split: 0 examples [00:00, ? examples/s]

In [9]:
print(xlsumEnDataset['test'][0]['title'])
print(xlsumEnDataset['test'][0]['summary'])
print(xlsumEnDataset['test'][0]['text'])

Trump inauguration speech: 'Angry', 'authentic', 'primal'
Donald Trump campaigned on becoming a president unlike any Washington has ever seen. With his inauguration speech, he's already set the tone.
By Kate DaileyBBC News Earlier this week, Trump posted a photo of himself sitting at a desk at Mar-a-Largo, a permanent marker hovering over a notepad. "Writing my inaugural address at the Winter White House, Mar-a-Lago, three weeks ago. Looking forward to Friday," he tweeted. Trump vows to end 'American carnage' Trump's angry call to arms Full text of Trump's inauguration speech It's unclear whether the president-elect actually wrote the speech himself, but the content was pure Trump: the same populist message that resonated throughout the primaries and the campaign. "Today, we are not merely transferring power from one administration to another, or from one party to another, but we are transferring power from Washington, DC, and giving it back to you, the people," he said at the beginnin

# Initializing tokenizer and Model

In [10]:
tokenizer = BartTokenizer.from_pretrained('facebook/bart-base')

# Since we are generating text for summerization, we shall be using the BartForConditionalGenration specifically
model0 = BartForConditionalGeneration.from_pretrained('facebook/bart-base')
model1 = BartForConditionalGeneration.from_pretrained('facebook/bart-base')
model2 = BartForConditionalGeneration.from_pretrained('facebook/bart-base')
model3 = BartForConditionalGeneration.from_pretrained('facebook/bart-base')

/usr/local/lib/python3.12/dist-packages/huggingface_hub/utils/_auth.py:93: UserWarning: 
The secret `HF_TOKEN` does not exist in your Colab secrets.
To authenticate with the Hugging Face Hub, create a token in your settings tab (https://huggingface.co/settings/tokens), set it as secret in your Google Colab and restart your session.
You will be able to reuse this secret in all of your notebooks.
Please note that authentication is recommended but still optional to access public models or datasets.
  warnings.warn(


vocab.json: 0.00B [00:00, ?B/s]

merges.txt: 0.00B [00:00, ?B/s]

tokenizer.json: 0.00B [00:00, ?B/s]

config.json: 0.00B [00:00, ?B/s]

model.safetensors:   0%|          | 0.00/558M [00:00<?, ?B/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/259 [00:00<?, ?it/s]

## Defining tokenize function for dataset tokenization

In [11]:
def tokenize(batch):
  # We shall truncate the input text to Bart's max length, which is 1024
  modelInput = tokenizer(batch['text'],padding='max_length',truncation=True,max_length=MAX_INPUT_LENGTH)
  # And truncate the summary to a length of 128
  labels = tokenizer(batch['summary'],padding='max_length',truncation=True,max_length=MAX_SUMMARY_LENGTH)

  modelInput['labels'] = labels['input_ids']
  return modelInput


## Actual mapping of tokenize function

In [12]:
testDs = xlsumEnDataset['test'].select(range(100)).map(tokenize,batched=True)
valDs = xlsumEnDataset['val'].select(range(100)).map(tokenize,batched=True)

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

Map:   0%|          | 0/100 [00:00<?, ? examples/s]

In [13]:
testDs.set_format('torch',columns=['input_ids','attention_mask','labels'])
valDs.set_format('torch',columns=['input_ids','attention_mask','labels'])

# Evaluate Model Function

In [14]:
def predict_model(model,testLoader,tokenizer, device):
  model.eval()

  generatedSummary = []
  originalSummary = []

  # setting torch to not use gradients
  with torch.no_grad():
    for batch in tqdm(testLoader, desc="Evaluating"):
      inputId = batch['input_ids'].to(device)
      attentionMask = batch['attention_mask'].to(device)
      labels = batch['labels'].to(device)

      # replace errors for padding
      labels = torch.where(labels != -100, labels, tokenizer.pad_token_id)

      # when generating summary, we set the max length of summary the same as initilised and bean search width to be 4 and we set early stopping to be true incase there is no improvement
      generatedId = model.generate(input_ids = inputId, attention_mask = attentionMask, max_length=MAX_SUMMARY_LENGTH, num_beams=BEAM_SEARCH_WIDTH, early_stopping=True)

      # decoding the original and generated inputIds
      decodedGeneratedSummary = tokenizer.batch_decode(generatedId,skip_special_tokens=True)
      decodedOriginalSummary = tokenizer.batch_decode(labels,skip_special_tokens=True)

      # Filter out any None values or empty strings to prevent errors in the evaluate library
      decodedGeneratedSummary = [s for s in decodedGeneratedSummary if s is not None and s.strip() != '']
      decodedOriginalSummary = [s for s in decodedOriginalSummary if s is not None and s.strip() != '']

      generatedSummary.extend(decodedGeneratedSummary)
      originalSummary.extend(decodedOriginalSummary)


  return generatedSummary, originalSummary

# Loading model, weights and dataset

In [15]:
torch.cuda.empty_cache()

## Loading Models and weights

---
Here Model 1 is the model with the following parameters:

MAX_SUMMARY_LENGTH = 128
MAX_INPUT_LENGTH = 1024
BEAM_SEARCH_WIDTH = 4

---

Model 2 (around 10000 parameters) and Model 3 (the entire dataset) are with the following parameters:

MAX_SUMMARY_LENGTH = 64
MAX_INPUT_LENGTH = 512
BEAM_SEARCH_WIDTH = 4

In [16]:
model1.load_state_dict(torch.load(os.path.join(modelSavePath,'Bart-Summarisation-Final-1.pth'), weights_only=True))
model2.load_state_dict(torch.load(os.path.join(modelSavePath,'Bart-Summarisation-Final-2-test.pth'), weights_only=True))
model3.load_state_dict(torch.load(os.path.join(modelSavePath,'Bart-Summarisation-Final-3.pth'), weights_only=True))

<All keys matched successfully>

## Loading Dataset

In [17]:
testLoader = torch.utils.data.DataLoader(testDs, batch_size=BATCH_SIZE, shuffle=False)
valLoader = torch.utils.data.DataLoader(valDs, batch_size=BATCH_SIZE, shuffle=False)

# Generating Sample predictions

In [18]:
model0.to(device)

preds0 = predict_model(model0,testLoader,tokenizer, device)

del model0

Evaluating: 100%|██████████| 1/1 [00:07<00:00,  7.96s/it]


In [19]:
model1.to(device)

preds1 = predict_model(model1,testLoader,tokenizer, device)

del model1

Evaluating: 100%|██████████| 1/1 [00:00<00:00,  1.26it/s]


In [20]:
model2.to(device)

preds2 = predict_model(model2,testLoader,tokenizer, device)

del model2

Evaluating: 100%|██████████| 1/1 [00:06<00:00,  6.39s/it]


In [21]:
model3.to(device)

preds3 = predict_model(model3,testLoader,tokenizer, device)

del model3

Evaluating: 100%|██████████| 1/1 [00:03<00:00,  3.62s/it]


In [22]:
print(preds0)
print(preds1)
print(preds2)
print(preds3)

(['By Kate DaileyBBC News Earlier this week, Trump posted a photo of himself sitting at a desk at Mar-a-Largo, a permanent marker hovering over a notepad. "Writing my inaugural address at the Winter White House, Mar-A-Lago, three weeks ago. Looking forward to', 'But Eluned Morgan conceded that it would be "difficult for us to stop" from a legal point of view. Her comments were criticised by a Labour AM. Alun Davies said threatening legal action "sounds like the last breath before you\'re thrown out of the pub". Mr Davies said he was', 'By Jon Welch and Paul MoseleyBBC News Details of health problems, family bereavements and personal issues were sent by the University of East Anglia (UEA) in Norwich to 298 students. Megan Baynes, 23, said she felt "sick and horrified" when she realised her details', 'By Helen BriggsBBC News In 2016, 1.9% of girls chose A-level physics, up from 1.6% in 2011. But that compared with 6.5% for boys in 2016 and 44% of schools in England still send no girls at

In [37]:
print(f'Predicted: {preds0[0][0]}')
print(f'Original: {preds0[1][0]}')

Predicted: By Kate DaileyBBC News Earlier this week, Trump posted a photo of himself sitting at a desk at Mar-a-Largo, a permanent marker hovering over a notepad. "Writing my inaugural address at the Winter White House, Mar-A-Lago, three weeks ago. Looking forward to
Original: Donald Trump campaigned on becoming a president unlike any Washington has ever seen. With his inauguration speech, he's already set the tone.


In [31]:
print(f'Predicted: {preds1[0]}')
print(f'Original: {preds1[1]}')

Predicted: []
Original: ["Donald Trump campaigned on becoming a president unlike any Washington has ever seen. With his inauguration speech, he's already set the tone.", 'The Welsh Government "would probably" take the UK government to court if it struck an international trade deal that tried to "trample on" the Welsh NHS, a minister said.', 'A university has mistakenly emailed hundreds of students intimate and sensitive personal information about dozens of fellow undergraduates.', 'Some progress has been made in encouraging girls to study physics at A-level, according to a report by the Institute of Physics (IoP).', 'The president of business organisation, the CBI, has said if politicians cannot "get their act together" on Brexit, then the only other option is to "go back to the people".', 'Ireland\'s health minister has condemned "reckless scenes" at a venue in Dublin which appeared to breach social distancing regulations.', "One tweeted picture saw Labour MP Emily Thornberry go from 

In [34]:
print(f'Predicted: {preds2[0][0]}')
print(f'Original: {preds2[1][0]}')

Predicted: By Kate DaileyBBC News Earlier this week, Trump posted a photo of himself sitting at a desk at Mar-a-Largo, a permanent marker hovering over a notepad. "Writing my inaugural address at the Winter White House, Mar-A-Lago, three weeks ago. Looking forward to
Original: Donald Trump campaigned on becoming a president unlike any Washington has ever seen. With his inauguration speech, he's already set the tone.


In [35]:
print(f'Predicted: {preds3[0][0]}')
print(f'Original: {preds3[1][0]}')

Predicted: US President-elect Donald Trump has delivered his inaugural address at Mar-a-Lago in Florida.
Original: Donald Trump campaigned on becoming a president unlike any Washington has ever seen. With his inauguration speech, he's already set the tone.
